# Publicador sintético — versão entregue aos participantes

Esta cópia permite **gerar e inspecionar localmente** exatamente o tipo de pacote usado na oficina, mas **não contém URL nem código de escrita no Firebase**.

Durante a aula, um único publicador controlado pelo instrutor alimenta o banco para evitar colisões entre várias publicações.


## 1. Configuração
Copie a **URL do Realtime Database** mostrada no console do Firebase. Não acrescente `/oficina/live.json`.

In [ ]:
MODO = "NORMAL"  # Seleciona o tipo de sinal/artefato a gerar.
DURACAO_SEGUNDOS = 8  # Duração usada apenas para a demonstração local.


In [ ]:
import time                         # Gera o instante de publicação usado no pacote sintético.
import uuid                         # Cria um identificador único para cada sessão de dados.
import matplotlib.pyplot as plt     # Permite visualizar os sinais sintéticos localmente.
import numpy as np                  # Gera e manipula numericamente os sinais ECG e PPG.

FS = 250  # Frequência de amostragem dos sinais sintéticos.
AMOSTRAS_POR_PACOTE = 50  # Quantidade de amostras geradas em cada pacote.
HR_BPM = 72  # Frequência cardíaca usada para gerar o exemplo sintético.
rng = np.random.default_rng(12345)  # Semente fixa torna os exemplos de ruído reproduzíveis.

MODOS_VALIDOS = [
    "NORMAL", "NOISE", "BASELINE", "MOTION",
    "SPIKE", "CLIPPING", "OUTLIER"
]

# Função: Gera uma forma gaussiana periódica usada para construir componentes do ECG sintético.
def _gauss_circular(fase, mu, sigma):
    d = np.minimum(np.abs(fase - mu), 1.0 - np.abs(fase - mu))
    return np.exp(-0.5 * (d / sigma) ** 2)

# Função: Gera amostras sintéticas sincronizadas de ECG e PPG para uma faixa de índices.
def gerar_sinais(sample_start, n, fs=FS, hr_bpm=HR_BPM):
    idx = np.arange(sample_start, sample_start + n)  # Cria os índices globais das amostras deste pacote.
    t = idx / fs  # Converte índice de amostra em tempo físico do sinal.
    periodo = 60.0 / hr_bpm  # Converte frequência cardíaca em segundos por batimento.
    fase = (t % periodo) / periodo  # Obtém a posição normalizada dentro de cada ciclo cardíaco.
    ecg = (
        0.08 * _gauss_circular(fase, 0.18, 0.025)
        - 0.12 * _gauss_circular(fase, 0.365, 0.012)
        + 1.00 * _gauss_circular(fase, 0.400, 0.010)
        - 0.25 * _gauss_circular(fase, 0.435, 0.014)
        + 0.30 * _gauss_circular(fase, 0.68, 0.055)
        + 0.015 * np.sin(2 * np.pi * 0.20 * t)
    )
    fase_ppg = ((t - 0.18) % periodo) / periodo  # Atrasa o PPG em relação ao ECG para representar o tempo de propagação do pulso.
    pulse = np.maximum(np.sin(np.pi * np.clip(fase_ppg / 0.55, 0, 1)), 0) ** 2
    pulse *= (fase_ppg < 0.55)
    dicrotic = 0.15 * _gauss_circular(fase_ppg, 0.62, 0.035)
    ppg_base = pulse + dicrotic
    ppg_ir = 50000 + 6500 * ppg_base + 120 * np.sin(2 * np.pi * 0.08 * t)
    ppg_red = 42000 + 4300 * ppg_base + 100 * np.sin(2 * np.pi * 0.08 * t + 0.2)
    return t, ecg.astype(float), ppg_ir.astype(float), ppg_red.astype(float)  # Garante que todos os sinais retornem como vetores de ponto flutuante.

# Função: Aplica ao sinal sintético o artefato selecionado para fins didáticos.
def aplicar_falha(modo, t, ecg, ppg_ir, ppg_red):
    modo = modo.upper()  # Padroniza o nome do modo para aceitar maiúsculas/minúsculas.
    if modo == "NORMAL":
        return ecg, ppg_ir, ppg_red
    if modo == "NOISE":
        ecg = ecg + rng.normal(0, 0.16, len(ecg))
        ppg_ir = ppg_ir + rng.normal(0, 700, len(ppg_ir))
        ppg_red = ppg_red + rng.normal(0, 550, len(ppg_red))
    elif modo == "BASELINE":
        ecg = ecg + 0.42 * np.sin(2 * np.pi * 0.27 * t)
    elif modo == "MOTION":
        artefato = 0.40 * np.sin(2 * np.pi * 1.3 * t) + rng.normal(0, 0.10, len(ecg))
        ecg = ecg + artefato
        ppg_ir = ppg_ir + 2500 * np.sin(2 * np.pi * 1.3 * t)
        ppg_red = ppg_red + 1800 * np.sin(2 * np.pi * 1.3 * t)
    elif modo == "SPIKE":
        mask = rng.random(len(ecg)) < 0.04  # Seleciona aleatoriamente cerca de 4% das amostras para receber picos artificiais.
        ecg = ecg.copy()  # Copia o vetor antes de alterar apenas algumas posições.
        ecg[mask] += rng.choice([-1.8, 1.8], size=mask.sum())  # Insere picos positivos/negativos nas posições escolhidas.
    elif modo == "CLIPPING":
        ecg = np.clip(ecg, -0.18, 0.55)  # Limita artificialmente a amplitude do ECG.
        ppg_ir = np.clip(ppg_ir, 49000, 54500)  # Limita artificialmente a amplitude do PPG IR.
    elif modo == "OUTLIER" and len(ecg):
        k = int(rng.integers(0, len(ecg)))  # Escolhe uma posição aleatória para inserir um valor extremo.
        ecg = ecg.copy()  # Copia o ECG antes de inserir o valor extremo.
        ppg_ir = ppg_ir.copy()  # Copia o PPG IR antes de inserir o valor extremo.
        ecg[k] = 4.5  # Valor extremo proposital no ECG.
        ppg_ir[k] = 90000  # Valor extremo proposital no PPG IR.
    return ecg, ppg_ir, ppg_red

# Função: Organiza sinais e metadados no mesmo formato de pacote usado pelos notebooks da oficina.
def montar_pacote(sample_start=0, seq=0, session_id=None, modo="NORMAL"):
    if session_id is None:
        session_id = uuid.uuid4().hex  # Gera um identificador aleatório único para a sessão sintética.
    t, ecg, ppg_ir, ppg_red = gerar_sinais(sample_start, AMOSTRAS_POR_PACOTE)
    ecg, ppg_ir, ppg_red = aplicar_falha(modo, t, ecg, ppg_ir, ppg_red)
    return {
        "session_id": session_id, "seq": seq, "sample_start": sample_start,
        "timestamp_pub": time.time(), "fs": FS, "n": AMOSTRAS_POR_PACOTE,
        "source": "SIM", "ecg": np.round(ecg, 5).tolist(),
        "ppg_ir": np.round(ppg_ir, 2).tolist(), "ppg_red": np.round(ppg_red, 2).tolist(),
    }


In [ ]:
pacote = montar_pacote(modo=MODO)
print("Campos do pacote:", list(pacote))
print("fs =", pacote["fs"], "Hz; n =", pacote["n"])
print("Primeiras amostras ECG:", pacote["ecg"][:5])


In [ ]:
t = np.arange(pacote["n"]) / pacote["fs"]  # Cria o tempo relativo dentro de um único pacote.
fig, ax = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
ax[0].plot(t, pacote["ecg"])  # Mostra o ECG sintético deste pacote.
ax[0].set_ylabel("ECG")
ax[1].plot(t, pacote["ppg_ir"], label="IR")
ax[1].plot(t, pacote["ppg_red"], label="Red")
ax[1].set_ylabel("PPG")
ax[1].set_xlabel("tempo no pacote (s)")
ax[1].legend()  # Exibe a legenda dos canais IR e Red.
plt.tight_layout()  # Ajusta os espaços da figura.
plt.show()  # Exibe a figura no notebook.


## Publicação no Firebase

**Desabilitada nesta cópia.** O notebook não contém URL do banco, credenciais, `requests.put()` nem `db.reference(...).set()`.
